In [3]:
# CLASSICAL BOOLEAN-BASED MULTI-SECRET SHARING SCHEME
# Exact implementation of Kapadiya et al. (2018) paper
# Generates meaningful shares using Boolean XOR operations

import cv2
import numpy as np
import os

# ========== CONFIGURATION ==========
TARGET_SIZE = (512, 512)  # Larger size for visible meaningful shares
n_secrets = 2             # Number of secret images (n)
# ===================================

print(f"📄 Classical Boolean-based Multi-Secret Sharing (Kapadiya et al. 2018)")
print(f"   Image size: {TARGET_SIZE[0]}x{TARGET_SIZE[1]}")
print(f"   Number of secrets: {n_secrets}")
print(f"   Total shares: {n_secrets + 1} (1 universal + {n_secrets} participant)\n")

def load_and_resize(img_path, demo_type='random'):
    """Load image and resize to target size."""
    for ext in ['.png', '.jpg', '.jpeg', '.tif', '.bmp']:
        test_path = img_path.replace(os.path.splitext(img_path)[1], ext)
        if os.path.exists(test_path):
            img = cv2.imread(test_path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                print(f"✅ Loaded {os.path.basename(test_path)}: {resized.shape}")
                return resized
    
    # Generate demo image
    print(f"⚠️  {img_path} not found, generating {demo_type} demo...")
    seed = sum(ord(c) for c in img_path) % 1000
    np.random.seed(seed)
    
    if demo_type == 'face':
        demo = np.full(TARGET_SIZE, 150, dtype=np.uint8)
        center = TARGET_SIZE[0] // 2
        radius = TARGET_SIZE[0] // 4
        cv2.circle(demo, (center, center), radius, 90, -1)
        cv2.circle(demo, (center - 20, center - 15), 8, 200, -1)
        cv2.circle(demo, (center + 20, center - 15), 8, 200, -1)
        cv2.ellipse(demo, (center, center + 10), (25, 15), 0, 0, 180, 80, -1)
    elif demo_type == 'boat':
        demo = np.full(TARGET_SIZE, 120, dtype=np.uint8)
        pts = np.array([[40, 80], [80, 80], [90, 100], [30, 100]], np.int32)
        cv2.fillPoly(demo, [pts], 200)
        cv2.rectangle(demo, (55, 40), (65, 80), 180, -1)
    elif demo_type == 'scene':
        demo = np.random.randint(80, 180, TARGET_SIZE).astype(np.uint8)
        cv2.rectangle(demo, (30, 50), (90, 110), 60, -1)
        cv2.circle(demo, (40, 40), 15, 220, -1)
    else:
        demo = np.random.randint(50, 200, TARGET_SIZE).astype(np.uint8)
    
    cv2.imwrite(img_path, demo)
    print(f"   Generated and saved to {img_path}")
    return demo

# Load images
print("\n📁 Loading images...")
secrets = []
for i in range(n_secrets):
    secret = load_and_resize(f'secret_{i+1}.tif', 'random')
    secrets.append(secret)

covers = []
cover_types = ['boat', 'face', 'scene', 'face']
for i in range(n_secrets + 1):
    cover_type = cover_types[i] if i < len(cover_types) else 'random'
    cover = load_and_resize(f'cover_{i+1}.tif', cover_type)
    covers.append(cover)

print("\n" + "="*60)
print("ENCODING PHASE - Generating Meaningful Shares")
print("="*60)

# Paper's Encoding Equations:
# S1[1:4] = G1[1:4] ⊕ G2[1:4] ⊕ ... ⊕ Gn[1:4]    (LSB 4 bits)
# S1[5:8] = C1[5:8]                              (MSB 4 bits)
# 
# Sn+1[1:4] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ Sn[5:8] ⊕ Gn[5:8]  (LSB 4 bits)
# Sn+1[5:8] = Cn+1[5:8]                                     (MSB 4 bits)

print("\n🔐 Step 1: Creating Universal Share (S1)...")
print("   Formula: S1[1:4] = G1[1:4] ⊕ G2[1:4] ⊕ ... ⊕ Gn[1:4]")
print("            S1[5:8] = C1[5:8]")

universal_share = np.zeros(TARGET_SIZE, dtype=np.uint8)
rows, cols = TARGET_SIZE

for i in range(rows):
    for j in range(cols):
        # XOR lower 4 bits of all secrets
        lsb_4 = 0
        for secret in secrets:
            lsb_4 ^= (secret[i, j] & 0x0F)
        
        # Take upper 4 bits from cover 1
        msb_4 = covers[0][i, j] & 0xF0
        
        # Combine
        universal_share[i, j] = lsb_4 | msb_4

# Save universal share
cv2.imwrite('universal_share.png', universal_share)
print(f"   ✅ Universal share created and saved")
print(f"      Should visually resemble cover_1.png (boat)")
print(f"      Pixel range: [{universal_share.min()}, {universal_share.max()}]")

# Calculate visual quality (PSNR with cover)
mse_cover = np.mean((covers[0].astype(float) - universal_share.astype(float)) ** 2)
psnr_cover = 20 * np.log10(255.0 / np.sqrt(mse_cover)) if mse_cover > 0 else float('inf')
print(f"      PSNR vs Cover 1: {psnr_cover:.2f} dB")

print(f"\n🔐 Step 2: Creating {n_secrets} Participant Shares...")

participant_shares = []
previous_share = universal_share.copy()

for k in range(n_secrets):
    print(f"\n   Participant Share {k+1} (S{k+2}):")
    print(f"      Formula: S{k+2}[1:4] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ S{k+1}[5:8] ⊕ G{k+1}[5:8]")
    print(f"               S{k+2}[5:8] = C{k+2}[5:8]")
    
    participant_share = np.zeros(TARGET_SIZE, dtype=np.uint8)
    
    for i in range(rows):
        for j in range(cols):
            # Start with universal share's upper 4 bits
            lsb_4 = (universal_share[i, j] & 0xF0) >> 4
            
            # XOR with previous participant shares' upper 4 bits
            for prev_share in participant_shares:
                lsb_4 ^= (prev_share[i, j] & 0xF0) >> 4
            
            # XOR with current secret's upper 4 bits
            lsb_4 ^= (secrets[k][i, j] & 0xF0) >> 4
            
            # Take upper 4 bits from corresponding cover
            msb_4 = covers[k+1][i, j] & 0xF0
            
            # Combine
            participant_share[i, j] = lsb_4 | msb_4
    
    participant_shares.append(participant_share)
    
    # Save participant share
    cv2.imwrite(f'participant_share_{k+1}.png', participant_share)
    print(f"      ✅ Saved participant_share_{k+1}.png")
    print(f"         Should visually resemble cover_{k+2}.png")
    print(f"         Pixel range: [{participant_share.min()}, {participant_share.max()}]")
    
    # PSNR with cover
    mse_cover = np.mean((covers[k+1].astype(float) - participant_share.astype(float)) ** 2)
    psnr_cover = 20 * np.log10(255.0 / np.sqrt(mse_cover)) if mse_cover > 0 else float('inf')
    print(f"         PSNR vs Cover {k+2}: {psnr_cover:.2f} dB")

print("\n" + "="*60)
print("✅ ENCODING COMPLETE!")
print("="*60)
print(f"\nGenerated {n_secrets + 1} meaningful shares:")
print("   - universal_share.png (looks like cover_1)")
for k in range(n_secrets):
    print(f"   - participant_share_{k+1}.png (looks like cover_{k+2})")
print("\nThese shares are visually meaningful and resemble cover images!")


📄 Classical Boolean-based Multi-Secret Sharing (Kapadiya et al. 2018)
   Image size: 512x512
   Number of secrets: 2
   Total shares: 3 (1 universal + 2 participant)


📁 Loading images...
✅ Loaded secret_1.tif: (512, 512)
✅ Loaded secret_2.tif: (512, 512)
✅ Loaded cover_1.tif: (512, 512)
✅ Loaded cover_2.tif: (512, 512)
✅ Loaded cover_3.tif: (512, 512)

ENCODING PHASE - Generating Meaningful Shares

🔐 Step 1: Creating Universal Share (S1)...
   Formula: S1[1:4] = G1[1:4] ⊕ G2[1:4] ⊕ ... ⊕ Gn[1:4]
            S1[5:8] = C1[5:8]
   ✅ Universal share created and saved
      Should visually resemble cover_1.png (boat)
      Pixel range: [0, 255]
      PSNR vs Cover 1: 31.87 dB

🔐 Step 2: Creating 2 Participant Shares...

   Participant Share 1 (S2):
      Formula: S2[1:4] = S1[5:8] ⊕ S2[5:8] ⊕ ... ⊕ S1[5:8] ⊕ G1[5:8]
               S2[5:8] = C2[5:8]
      ✅ Saved participant_share_1.png
         Should visually resemble cover_2.png
         Pixel range: [0, 223]
         PSNR vs Cover 2: 31